# Bordeaux Overture Buildings From Fused Mirror

Query the Fused Source Cooperative mirror directly with DuckDB. This avoids the public Fused display UDF, which can return tile/chunk summary polygons instead of individual building footprints.

In [1]:
import json
from pathlib import Path

import duckdb
import geopandas as gpd
from shapely.geometry import shape

In [2]:
release = '2025-06-25-0'
stac_path = Path('/media/disk1/vantor_bordeaux_data/B110002101F01500.json')
dst = Path('/media/disk2') / 'bordeaux_overture_buildings_2025-06-25.geojson'

stac = json.loads(stac_path.read_text())
minx, miny, maxx, maxy = stac['bbox']
image_footprint = shape(stac['geometry'])

base = f's3://us-west-2.opendata.source.coop/fused/overture/{release}/theme=buildings/type=building'
building_paths = [f'{base}/part={part}/**/*.parquet' for part in range(6)]

building_paths

['s3://us-west-2.opendata.source.coop/fused/overture/2025-06-25-0/theme=buildings/type=building/part=0/**/*.parquet',
 's3://us-west-2.opendata.source.coop/fused/overture/2025-06-25-0/theme=buildings/type=building/part=1/**/*.parquet',
 's3://us-west-2.opendata.source.coop/fused/overture/2025-06-25-0/theme=buildings/type=building/part=2/**/*.parquet',
 's3://us-west-2.opendata.source.coop/fused/overture/2025-06-25-0/theme=buildings/type=building/part=3/**/*.parquet',
 's3://us-west-2.opendata.source.coop/fused/overture/2025-06-25-0/theme=buildings/type=building/part=4/**/*.parquet',
 's3://us-west-2.opendata.source.coop/fused/overture/2025-06-25-0/theme=buildings/type=building/part=5/**/*.parquet']

In [3]:
con = duckdb.connect()
con.execute('INSTALL httpfs')
con.execute('LOAD httpfs')
con.execute("SET s3_region='us-west-2'")
con.execute("SET s3_endpoint='s3.us-west-2.amazonaws.com'")
con.execute("SET s3_url_style='path'")

sql = '''
SELECT id, geometry
FROM read_parquet(?, hive_partitioning = 1, union_by_name = true)
WHERE bbox.xmax >= ? AND bbox.xmin <= ?
  AND bbox.ymax >= ? AND bbox.ymin <= ?
'''

df = con.execute(sql, [building_paths, minx, maxx, miny, maxy]).fetch_df()
print(f'Candidate rows from bbox filter: {len(df):,}')
df.head()

Candidate rows from bbox filter: 165,142


,id,geometry
0,81be899f-4fbc-4469-9210-d2a77d55be15,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 5, 0, 0, 0, 103, 1..."
1,201344c2-74fb-4d8c-af53-717382c7f2eb,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 5, 0, 0, 0, 82, 71..."
2,71660b4f-3661-4724-bbff-27a4cf4a4087,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 7, 0, 0, 0, 124, 5..."
3,6169f5c5-3344-42fa-83a8-8ad8955addcc,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 5, 0, 0, 0, 230, 1..."
4,af556082-5862-4491-b6b1-82602dd4d0a4,"[1, 3, 0, 0, 0, 1, 0, 0, 0, 5, 0, 0, 0, 25, 16..."


In [4]:
if df.empty:
    raise ValueError('No Overture building rows matched the Bordeaux image bbox')

wkb = df.pop('geometry').map(lambda value: bytes(value) if isinstance(value, bytearray) else value)
geometry = gpd.GeoSeries.from_wkb(wkb, crs='EPSG:4326')
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

# Keep real building polygons whose centroids fall inside the actual Vantor image footprint.
gdf = gdf[gdf.geometry.centroid.within(image_footprint)].reset_index(drop=True)

print(f'Buildings inside image footprint: {len(gdf):,}')
print(gdf.geom_type.value_counts())
print(gdf.total_bounds)

dst.unlink(missing_ok=True)
gdf.to_file(dst, driver='GeoJSON')
dst

/tmp/ipykernel_3411964/3559778738.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf = gdf[gdf.geometry.centroid.within(image_footprint)].reset_index(drop=True)


Buildings inside image footprint: 122,648
Polygon         122640
MultiPolygon         8
Name: count, dtype: int64
[-1.2594125 44.5694188 -1.0401558 45.1164601]


PosixPath('/media/disk2/bordeaux_overture_buildings_2025-06-25.geojson')